### Imports

In [1]:
# the "> /dev/null 2>&1" at the end of each line simply suppresses the 
# installation log
!pip3 install rocketcea > /dev/null 2>&1 
# in case it's not installed already (will be used to model NASA CEA)
!pip3 install rocketisp > /dev/null 2>&1 
# used to model chamber and nozzle losses
!pip3 install ambiance > /dev/null 2>&1 
# this is the standard atmosphere model that we will use 
# later in the notebook
!pip3 install cantera > /dev/null 2>&1 
# thermophysical, fluids, and kinetics toolbox
!pip3 install ipywidgets > /dev/null 2>&1 
# for the engien design widget at the end

In [2]:
from rocketcea.cea_obj_w_units import CEA_Obj

In [14]:
from rocketcea.cea_obj_w_units import CEA_Obj
import numpy as np
import matplotlib.pyplot as plt
from ambiance import Atmosphere
import multiprocessing
from rocketisp.rocket_isp import RocketThruster
from rocketisp.geometry import Geometry
from rocketisp.stream_tubes import CoreStream
from rocketisp.efficiencies import Efficiencies
from rocketisp.nozzle.nozzle import Nozzle
from rocketisp.geometry import solidCylVol, solidFrustrumVol
from scipy.optimize import minimize
from scipy.optimize import fsolve
from scipy.interpolate import griddata
import ipywidgets as widgets
from IPython.display import display
import cantera as ct
from PIL import Image
import csv

In [16]:
rocket = CEA_Obj(oxName='LOX', fuelName='RP-1', temperature_units='degK', 
                 cstar_units='m/sec', specific_heat_units='kJ/kg degK', 
                 sonic_velocity_units='m/s', enthalpy_units='J/kg', 
                 density_units='kg/m^3')
# This rocket object will be used going forward
# This initialization assumes an inifite area combustor. 
# This is will later be changed

In [17]:
# pip install CoolProp numpy
from CoolProp.CoolProp import PropsSI
import numpy as np

### Equivalence Ratio

In [18]:
def ER(fMM, oMM, fCoefA, oCoef, MR):
    MRStoich = (oCoef * oMM) / (fCoefA * fMM)
    return MRStoich / MR

### Reactants

In [19]:
fuel = 'C12H26'
oxidizer = 'O2'
fMM = 170.32
oMM = 32
fCoefA = 1
oCoef = 18.5
MR = 2

In [20]:
ER = ER(fMM, oMM, fCoefA, oCoef, 2)
print(ER)

1.737905119774542


In [21]:
fCoefS = ER
C = 12
H = 26
O = 2
nC = fCoefS * C
nH = fCoefS * H
nO = oCoef * O

### Products

In [22]:
#Reactant moles
n1 = 0
n2 = 0
n3 = 0

#### CO, H2, OH

In [23]:
#        0      1      2
# X = [X_CO,   X_H2,  X_OH]
def linear_equations_X_1(X):
    eq1 = np.sum(X) - 1
    eq2 = X[0] - 2*(nC/nH)*X[1] - (nC/nH)*X[2]
    eq3 = -(nH/nO)*X[0] + 2*X[1] + (1-(nH/nO))*X[2]
    return [eq1, eq2, eq3]

def linear_equations_n_1(X):
    eq1 = X[0] - nC
    eq2 = 2*X[1] + X[2] - nH
    eq3 = X[0] + X[2] - nO
    return [eq1, eq2, eq3]

X0 = [1, 1, 1]
X = fsolve(linear_equations_X_1, X0)
print(f"X | CO: {X[0]:.4f}, H2: {X[1]:.4f}, OH: {X[2]:.4f}")

n0 = [1, 1, 1]
[n1, n2, n3] = fsolve(linear_equations_n_1, n0)
print(f"n | CO: {n1:.4f} mol, H2: {n2:.4f} mol, OH: {n3:.4f} mol")


X | CO: 0.4048, H2: 0.2818, OH: 0.3134
n | CO: 20.8549 mol, H2: 14.5202 mol, OH: 16.1451 mol


#### CO2, H2O, CO

In [24]:
#        0      1      2
# X = [X_CO2, X_H2O,  X_CO]
# def linear_equations_X_2(X):
#     eq1 = np.sum(X) - 1
#     eq2 = X[0] - 2*(nC/nH)*X[1] - (nC/nH)*X[2]
#     eq3 = -(nH/nO)*X[0] + 2*X[1] + (1-(nH/nO))*X[2]
#     return [eq1, eq2, eq3]

def linear_equations_n_2(X):
    eq1 = X[0] + X[2] - nC
    eq2 = 2*X[1] - nH
    eq3 = 2*X[0] + X[1] + X[2] - nO
    return [eq1, eq2, eq3]

# X0 = [1, 1, 1]
# X = fsolve(linear_equations_X_2, X0)
# print(f"CO2: {X[0]:.4f}, H2O: {X[1]:.4f}, CO: {X[2]:.4f}")

n0 = [1, 1, 1]
[n1, n2, n3] = fsolve(linear_equations_n_2, n0)
print(f"n | CO2: {n1:.4f} mol, H2o: {n2:.4f} mol, CO: {n3:.4f} mol")

# DOES NOT BALANCE


n | CO2: -6.4476 mol, H2o: 22.5928 mol, CO: 27.3025 mol


#### CO, H2O, H2

In [27]:
#        0       1       2
# X = [X_H2O,   X_CO,  X_H2]
def linear_equations_X_3(X):
    eq1 = np.sum(X) - 1
    eq2 = X[1] - 2*(nC/nH)*X[0] - 2*(nC/nH)*X[2]
    eq3 = 2*X[0] + 2*X[2] - (nH/nO)*X[1] - (nH/nO)*X[0] 
    return [eq1, eq2, eq3]

def linear_equations_n_3(X):
    eq1 = X[1] - nC
    eq2 = 2*X[0] + 2*X[2] - nH
    eq3 = X[0] + X[1] - nO
    return [eq1, eq2, eq3]

X0 = [1, 1, 1]
X = fsolve(linear_equations_X_3, X0)
print(f"X | H2O: {X[0]:.4f}, CO: {X[1]:.4f}, H2: {X[2]:.4f}")

n0 = [1, 1, 1]
[n1, n2, n3] = fsolve(linear_equations_n_3, n0)
print(f"n | H2O: {n1:.4f} mol, CO: {n2:.4f} mol, H2: {n3:.4f} mol")


X | H2O: 0.3716, CO: 0.4800, H2: 0.1484
n | H2O: 16.1451 mol, CO: 20.8549 mol, H2: 6.4476 mol


#### CO, H2, CO2

In [36]:
#        0       1       2
# X = [X_CO,   X_H2,  X_CO2]
def linear_equations_X_4(X):
    eq1 = np.sum(X) - 1
    eq2 = X[0] + X[2] - 2*(nC/nH)*X[1]
    eq3 = 2*X[1] - (nH/nO)*X[0] - 2*(nH/nO)*X[2]
    return [eq1, eq2, eq3]

def linear_equations_n_4(X):
    eq1 = X[0] + X[2] - nC
    eq2 = 2*X[1] - nH
    eq3 = X[0] + 2*X[2] - nO
    return [eq1, eq2, eq3]

X0 = [1, 1, 1]
X = fsolve(linear_equations_X_4, X0)
print(f"X | CO: {X[0]:.4f}, H2: {X[1]:.4f}, CO2: {X[2]:.4f}")

n0 = [1, 1, 1]
[n1, n2, n3] = fsolve(linear_equations_n_4, n0)
print(f"n | CO: {n1:.4f} mol, H2: {n2:.4f} mol, CO2: {n3:.4f} mol")

X | CO: 0.1084, H2: 0.5200, CO2: 0.3716
n | CO: 4.7097 mol, H2: 22.5928 mol, CO2: 16.1451 mol


In [37]:
#Check moles
print(n1)
print(n2)
print(n3)

4.709722874589016
22.592766557069048
16.14513856270549


In [38]:
n1MM = 0.028
n2MM = 0.002
n3MM = 0.044

### Enthalpy

#### Enthalpy Reactants

In [39]:
#Enthalpy units: kJ/kg
Tref = 298.15

fEForm = -1.45e2
fT = 298.15
fESens = (PropsSI('H', 'T', fT, 'P', 1.0e5, 'n-Dodecane') - PropsSI('H', 'T', Tref, 'P', 1.0e5, 'n-Dodecane')) / 1000

oEForm = 0
oT = 90.17
oESens = (PropsSI('H', 'T', oT, 'P', 1.0e5, 'Oxygen') - PropsSI('H', 'T', Tref, 'P', 1.0e5, 'Oxygen')) / 1000


print(fESens)
print(oESens)

0.0
-191.31206460637947


In [40]:
fH = fEForm + fESens
oH = oEForm + oESens

print(fH)
print(oH)

Ein = (fCoefS * fMM / 1000 * fH) + (oCoef * oMM / 1000 * oH)
print(Ein)

-145.0
-191.31206460637947
-156.17674224697663


#### Enthalpy Products

In [41]:
n1EForm = -3.95e3
n2EForm = 0
n3EForm = -8.94e3

n1M = n1 * n1MM
n2M = n2 * n2MM
n3M = n3 * n3MM

n1Name = 'CarbonMonoxide'
n2Name = 'Hydrogen'
n3Name = 'CarbonDioxide'

In [42]:
EFormOut = (n1M * n1EForm) + (n2M * n2EForm) + (n3M * n3EForm)

TARGET_KJ = Ein - EFormOut

print(TARGET_KJ)

6715.5703127084


### Temperature

In [43]:
# ---------- sensible enthalpy relative to 298.15 K using CoolProp ----------
def h_sensible_kJkg_coolprop(fluid: str, T: float, p_Pa: float = 1.0e5) -> float:
    """Return sensible enthalpy h(T)-h(298.15 K) in kJ/kg for a CoolProp fluid."""
    hT   = PropsSI('H', 'T', T,        'P', p_Pa, fluid)  # J/kg
    href = PropsSI('H', 'T', 298.15,   'P', p_Pa, fluid)  # J/kg
    return (hT - href) / 1000.0  # kJ/kg

# ---------- residual function for the energy balance ----------
def residual(T: float) -> float:
    h_n1  = h_sensible_kJkg_coolprop(n1Name, T)  # kJ/kg
    h_n2  = h_sensible_kJkg_coolprop(n2Name, T)        # kJ/kg
    h_n3  = h_sensible_kJkg_coolprop(n3Name, T)                          # kJ/kg
    return (n1M*h_n1 + n2M*h_n2 + n3M*h_n3) - TARGET_KJ

# ---------- bisection solver ----------
def solve_T_bisection(T_lo: float = 1000.0, T_hi: float = 5000.0, tolT: float = 1e-3, maxit: int = 200):
    f_lo = residual(T_lo)
    f_hi = residual(T_hi)
    if f_lo * f_hi > 0:
        raise RuntimeError(f"Bracket does not change sign: F({T_lo})={f_lo:.3f}, F({T_hi})={f_hi:.3f}")
    a, b, fa, fb = T_lo, T_hi, f_lo, f_hi
    for _ in range(maxit):
        c = 0.5*(a + b)
        fc = residual(c)
        if abs(fc) < 1e-3 or (b - a) < tolT:
            return c, fc
        if fa * fc <= 0:
            b, fb = c, fc
        else:
            a, fa = c, fc
    return c, fc

if __name__ == "__main__":
    T_f, res = solve_T_bisection()
    print(f"Adiabatic flame temperature ≈ {T_f:.1f} K  (residual {res:.3f} kJ)")


Adiabatic flame temperature ≈ 3896.8 K  (residual 0.000 kJ)


##### OH Solver (Not in CoolProp)

In [ ]:
MW_OH = 0.017007  # kg/mol
T_pts = np.array([298.15, 300, 500, 800, 1000, 1200, 1600, 2000, 2500, 3000, 3500, 4000], dtype=float)
y_kJ_per_kmol = np.array([0, 0.055, 6.079, 15.467, 21.888, 28.432, 41.838, 55.506, 72.827, 90.315, 107.89, 125.50], dtype=float)

def h_sensible_kJkg_OH(T: float) -> float:
    """Sensible enthalpy for OH in kJ/kg using your tabulated (h_T - h_298) [kJ/kmol]."""
    # linear interpolation over your table
    h_kJ_per_kmol = np.interp(T, T_pts, y_kJ_per_kmol)
    return h_kJ_per_kmol / MW_OH  # → kJ/kg

### RocketCEA

#### Mole Fractions

In [11]:
ispObj = CEA_Obj(oxName='LOX', fuelName='RP1')

def print_mole_fraction_table(ispObj, Pc_psia=300.0, MR=2.0, eps=40.0,
                              frozen=0, frozenAtThroat=0, digits=5):
    molWtD, moleFracD = ispObj.get_SpeciesMoleFractions(
        Pc=Pc_psia, MR=MR, eps=eps, frozen=frozen, frozenAtThroat=frozenAtThroat
    )

    # Build rows: (species, [chamber, throat, exit@eps, exit@deliv], MW)
    rows = []
    for species, mf_list in moleFracD.items():
        rows.append((species, mf_list, molWtD[species]))

    # ---- sort by CHAMBER mole fraction (index 0) descending ----
    rows.sort(key=lambda r: r[1][0], reverse=True)

    # ---- pretty print with headers and index column ----
    header = (
        f"ROCKETCEA MOLE FRACTIONS  (frozen={frozen}, frozenAtThroat={frozenAtThroat}, "
        f"Pc={Pc_psia} psia, MR={MR}, eps={eps})"
    )
    print(header)
    print("-" * len(header))

    # Column titles
    col_titles = [
        "#", "Species", "Chamber", "Throat", "Nozzle Exit", "Pressure-Matched Exit", "MW (g/mol)"
    ]
    print(f"{col_titles[0]:>3}  {col_titles[1]:<12}  {col_titles[2]:>10}  {col_titles[3]:>10}  "
          f"{col_titles[4]:>13}  {col_titles[5]:>21}  {col_titles[6]:>10}")

    # Row format
    fmt = f"{{:>3}}  {{:<12}}  {{:>{10+digits-5}.{digits}f}}  {{:>{10+digits-5}.{digits}f}}  " \
          f"{{:>{13+digits-5}.{digits}f}}  {{:>{21+digits-5}.{digits}f}}  {{:>10.4f}}"

    # Print rows with index
    for i, (sp, mfs, mw) in enumerate(rows, start=1):
        print(fmt.format(i, sp, mfs[0], mfs[1], mfs[2], mfs[3], mw))

    print("=" * len(header))
    print()

# ---- Run the three cases exactly like your original loop ----
for frozen, frozenAtThroat in [(0,0), (1,0), (1,1)]:
    print_mole_fraction_table(
        ispObj, Pc_psia=300.0, MR=2.0, eps=40.0, frozen=frozen, frozenAtThroat=frozenAtThroat, digits=5
    )


ROCKETCEA MOLE FRACTIONS  (frozen=0, frozenAtThroat=0, Pc=300.0 psia, MR=2.0, eps=40.0)
---------------------------------------------------------------------------------------
  #  Species          Chamber      Throat    Nozzle Exit  Pressure-Matched Exit  MW (g/mol)
  1  *CO              0.41666     0.41666        0.41671                0.31967     28.0101
  2  H2O              0.26844     0.26844        0.27648                0.19088     18.0153
  3  *H2              0.18537     0.18537        0.18978                0.30181      2.0159
  4  *CO2             0.07649     0.07649        0.08119                0.18764     44.0095
  5  *H               0.03100     0.03100        0.02247                0.00000      1.0079
  6  *OH              0.01924     0.01924        0.01210                0.00000     17.0073
  7  *O               0.00177     0.00177        0.00080                0.00000     15.9994
  8  *O2              0.00100     0.00100        0.00046                0.00000     31.9

#### Critical Values

In [4]:
# rp1_lox_frozen_table_rcea121.py
# Works with RocketCEA 1.2.1 method names/signatures.

from rocketcea.cea_obj import CEA_Obj

# -------- inputs --------
Pc_psia    = 300.0
Pe_psia    = 10.0
Pc_over_Pe = Pc_psia / Pe_psia
MR         = 2.0
cstar_eff  = 0.92
g0         = 9.80665  # m/s^2
R_to_K     = 5.0/9.0  # degR -> K

# ---- CEA setup ----
cea = CEA_Obj(oxName='LOX', fuelName='RP-1')

# ---- area ratio Ae/At such that Pc/Pe = 30 (frozen) ----
# get_eps_at_PcOvPe exists in some builds, but bisection on get_PcOvPe is universal.
# ---- area ratio Ae/At such that Pc/Pe = 30 (frozen) ----
lo, hi = 1.01, 100.0  # reasonable bracket
for _ in range(60):
    mid = 0.5*(lo + hi)
    ratio = cea.get_PcOvPe(Pc=Pc_psia, MR=MR, eps=mid, frozen=1)
    if ratio > Pc_over_Pe:
        # ratio too large -> eps too large -> move upper bound down
        hi = mid
    else:
        # ratio too small -> eps too small -> move lower bound up
        lo = mid
eps = 0.5*(lo + hi)


# ---- chamber (equilibrium) ----
# Fast way to get Tcomb (degR), chamber MW (lbm/lbmole), chamber gamma:
IspVac_eq, Cstar_ftps, Tcomb_R, MW_c_lbmpermole, gamma_c = cea.get_IvacCstrTc_ChmMwGam(
    Pc=Pc_psia, MR=MR, eps=eps
)
T_c = Tcomb_R * R_to_K                          # K
MW_c = MW_c_lbmpermole             # -> kg/kmol (lbm/lbmole * 0.45359237)
# Note: 1 lbm/lbmole == 1 g/mol; 1 g/mol = 1 kg/kmol; 1 lbm ≈ 0.45359237 kg

# ---- exit (frozen) ----
# Exit MW & gamma directly (uses 'exit' lowercase in 1.2.1):
MW_e_lbmpermole, gamma_e = cea.get_exit_MolWt_gamma(
    Pc=Pc_psia, MR=MR, eps=eps, frozen=1, frozenAtThroat=0
)
MW_e = MW_e_lbmpermole             # kg/kmol

# Exit Mach number:
M_e = cea.get_MachNumber(Pc=Pc_psia, MR=MR, eps=eps, frozen=1, frozenAtThroat=0)

# Temperatures (degR) at chamber/throat/exit; convert to K. With frozen=1 this is the frozen exit T.
Tch_R, Tth_R, Tex_R = cea.get_Temperatures(Pc=Pc_psia, MR=MR, eps=eps, frozen=1, frozenAtThroat=0)
T_e = Tex_R * R_to_K

# ---- performance (frozen) ----
Isp_ideal = cea.get_Isp(Pc=Pc_psia, MR=MR, eps=eps, frozen=1, frozenAtThroat=0)  # s (vac, ideal)
Isp = cstar_eff * Isp_ideal
Ve  = Isp * g0

# ---- output ----
print("\n=== RP-1/LOX Frozen-Flow (Pc=%.1f psia, Pe=%.1f psia, MR=%.2f) ==="
      % (Pc_psia, Pe_psia, MR))
print("Pc/Pe:                                   %.3f" % Pc_over_Pe)
print("Area ratio eps (Ae/At):                  %.5f" % eps)

print("\n-- Chamber (Equilibrium) --")
print("Adiabatic flame temperature T_c [K]:     %.2f" % T_c)
print("Chamber molecular weight MW_c [kg/kmol]: %.4f" % MW_c)
print("Chamber gamma_c [-]:                     %.5f" % gamma_c)

print("\n-- Exit (Frozen) --")
print("Exit gamma_e [-]:                         %.5f" % gamma_e)
print("Exit MW_e [kg/kmol]:                     %.4f" % MW_e)
print("Exit Mach number M_e [-]:                %.4f" % M_e)
print("Exit temperature T_e [K]:                %.2f" % T_e)

print("\n-- Performance (with C* eff = %.0f%%) --" % (100*cstar_eff))
print("Specific impulse Isp [s]:                %.3f" % Isp)
print("Exhaust velocity Ve [m/s]:               %.2f" % Ve)
print("\n(Ideal frozen Isp @ eps) [s]:            %.3f" % Isp_ideal)



=== RP-1/LOX Frozen-Flow (Pc=300.0 psia, Pe=10.0 psia, MR=2.00) ===
Pc/Pe:                                   30.000
Area ratio eps (Ae/At):                  4.50789

-- Chamber (Equilibrium) --
Adiabatic flame temperature T_c [K]:     3257.01
Chamber molecular weight MW_c [kg/kmol]: 20.6665
Chamber gamma_c [-]:                     1.16171

-- Exit (Frozen) --
Exit gamma_e [-]:                         1.26227
Exit MW_e [kg/kmol]:                     20.6665
Exit Mach number M_e [-]:                2.7770
Exit temperature T_e [K]:                1668.81

-- Performance (with C* eff = 92%) --
Specific impulse Isp [s]:                264.471
Exhaust velocity Ve [m/s]:               2593.57

(Ideal frozen Isp @ eps) [s]:            287.468


#### RocketCEA Frozen Flow Mole Fractions

|  # | Species | Chamber |  Throat | Nozzle Exit | Second Exit | MW (g/mol) |
| -: | :------ | ------: | ------: | ----------: | --------------------: | ---------: |
|  1 | *CO     | 0.41666 | 0.41666 |     0.41671 |               0.41671 |    28.0101 |
|  2 | H2O     | 0.26844 | 0.26844 |     0.27648 |               0.27648 |    18.0153 |
|  3 | *H2     | 0.18537 | 0.18537 |     0.18978 |               0.18978 |     2.0159 |
|  4 | *CO2    | 0.07649 | 0.07649 |     0.08119 |               0.08119 |    44.0095 |
|  5 | *H      | 0.03100 | 0.03100 |     0.02247 |               0.02247 |     1.0079 |
|  6 | *OH     | 0.01924 | 0.01924 |     0.01210 |               0.01210 |    17.0073 |
|  7 | *O      | 0.00177 | 0.00177 |     0.00080 |               0.00080 |    15.9994 |
|  8 | *O2     | 0.00100 | 0.00100 |     0.00046 |               0.00046 |    31.9988 |
|  9 | HCO     | 0.00002 | 0.00002 |     0.00001 |               0.00001 |    29.0180 |


### Table of Results

##### Mole Fractions

| Species |     CEA | CO, H2O, H2 | % Error | CO, H2O, CO2 | % Error | CO, H2, CO2 | % Error | CO, H2, OH | % Error |
| :------ | ------: | :---------: | :-----: | :----------: | :-----: | :---------: | :-----: | :--------: | :-----: |
| CO      | 0.41666 |      0.4800      |    15.20%    |       NA      |    NA    |      0.1084      |    -73.98%    |      0.4048     |    -2.85%    |
| H2O     | 0.26844 |      0.3716      |    38.43%    |       NA      |    NA    |      ~      |    ~    |      ~     |    ~    |
| H2      | 0.18537 |      0.1484      |    -19.94%    |       ~      |    ~    |      0.5200      |    180.52%    |      0.2818     |    52.02%    |
| CO2     | 0.07649 |      ~      |    ~    |       NA      |    NA    |      0.3716      |    386%    |      ~     |    ~    |
| H       | 0.03100 |      ~      |    ~    |       ~      |    ~    |      ~      |    ~    |      ~     |    ~    |
| OH      | 0.01924 |      ~      |    ~    |       ~      |    ~    |      ~      |    ~    |      0.3134     |    1529%    |
| O       | 0.00177 |      ~      |    ~    |       ~      |    ~    |      ~      |    ~    |      ~     |    ~    |
| O2      | 0.00100 |      ~      |    ~    |       ~      |    ~    |      ~      |    ~    |      ~     |    ~    |
| HCO     | 0.00002 |      ~      |    ~    |       ~      |    ~    |      ~      |    ~    |      ~     |    ~    |


##### Critical Values

| Quantity                         |     CEA | CO, H2O, H2 | % Error  | CO, H2, CO2 |     % Error | CO, H2, OH | % Error |
| :------------------------------- | ------: | ----------: | ------: | ----------: | ----------: | ---------: | ------: |
| Adiabatic Flame Temperature (K)  | 3257.01 |      3404.6 |  +4.53% |  3896.8 | +19.65% |     1253.7 | −61.51% |
| Chamber Molecular Weight (g/mol) |   20.67 |       20.44 |  −1.11% |   20.44 |  −1.12% |      17.23 | −16.64% |
| Chamber Specific Heat Ratio      |   1.162 |       1.228 |  +5.68% |   1.252 |  +7.74% |      1.325 | +14.03% |
| Exhaust Mach Number              |   2.777 |       2.779 |  +0.07% |   2.791 |  +0.50% |      2.832 |  +1.98% |
| Exhaust Temperature (K)          | 1668.81 |        1809 |  +8.40% |  1967.7 | +17.91% |        545 | −67.34% |
| Exhaust Velocity (m/s)           | 2593.57 |        2642 |  +1.87% |    2793 |  +7.69% |       1671 | −35.57% |
| Specific Impulse (s)             | 264.471 |         269 |  +1.71% |   284.8 |  +7.69% |        170 | −35.72% |
